In [3]:
import scanpy as sc
import pandas as pd
import numpy as np
import glob
import os
import re

# ==============================================================================
# 1. SETUP & DEFINITIONS
# ==============================================================================

# Path to your metadata
META_PATH = "/Users/adiallo/Desktop/Thesis/Data_Documents/data_all.csv"

print(f"Loading metadata from: {META_PATH}")
meta = pd.read_csv(META_PATH)
# Ensure deident is string to match filenames
meta["deident"] = meta["deident"].astype(str)

# Define the h6 Mapping Logic (Granular Immune + Structure)
def map_to_h6_like(col_name):
    # Remove standard prefix
    clean = col_name.replace("meanscell_abundance_w_sf_", "")
    
    # --- TUMOR ---
    if clean.startswith("Tumor"): return "Tumor"
    
    # --- EPITHELIAL (Normal) ---
    if clean.startswith("cE"): return "Epithelial_Normal"
        
    # --- STRUCTURAL (Endothelial / Stromal) ---
    if clean.startswith("cS"):
        try:
            # Extract number (e.g., 'cS01' -> 1)
            num_match = re.search(r'\d+', clean)
            if num_match:
                num = int(num_match.group())
                if 1 <= num <= 14: return "Endothelial"
                if num == 32: return "Smooth_Muscle"
                return "Fibroblast_Stromal"
        except:
            return "Fibroblast_Stromal" # Fallback
            
    # --- B CELLS & PLASMA ---
    if clean.startswith("cB"): return "B_Cell" 
    if clean.startswith("cP"): return "Plasma"

    # --- MYELOID ---
    if clean.startswith("cM"):
        if "cM10" in clean: return "Neutrophil_Granulocyte"
        if "cM01" in clean or "cM02" in clean: return "Mono_Macrophage"
        return "Dendritic_Cell" # cM03-cM09
    if clean.startswith("cMA"): return "Mast"

    # --- T CELLS & NK ---
    if clean.startswith("cTNI"):
        try:
            num_match = re.search(r'\d+', clean)
            if num_match:
                num = int(num_match.group())
                if 1 <= num <= 7: return "CD4_T"
                if 8 <= num <= 9: return "Treg"
                if 10 <= num <= 16: return "CD8_T"
                if 17 <= num <= 22: return "T_Other_GammaDelta"
                if 23 <= num <= 25: return "NK"
                if num == 26: return "ILC"
        except: 
            return "T_Other"

    return "Other"

# ==============================================================================
# 2. FILE DISCOVERY
# ==============================================================================

valid_files = []
print("\nScanning for valid files...")

# Assuming the .h5ad files are in the CURRENT working directory
# If they are elsewhere, change glob.glob("/path/to/files/*.h5ad")
for h5_file in glob.glob("*.h5ad"):
    fname = os.path.basename(h5_file)
    sid = fname.replace(".h5ad","").replace("_adata","")
    
    if "atlas" in fname.lower() or "reference" in fname.lower(): continue
    if sid not in set(meta["deident"]): continue
    
    # Quick check without full load
    try:
        adata_peek = sc.read_h5ad(h5_file, backed='r')
        if "means_cell_abundance_w_sf" in adata_peek.obsm.keys():
            valid_files.append(h5_file)
    except:
        continue

print(f"Found {len(valid_files)} valid files to process.")

# ==============================================================================
# 3. MAIN LOOP: CALCULATE PROPORTIONS
# ==============================================================================

results_list = []

print("\nStarting batch processing...")

for i, filepath in enumerate(valid_files):
    sid = os.path.basename(filepath).replace(".h5ad","").replace("_adata","")
    
    try:
        # A. Load Data
        adata = sc.read_h5ad(filepath)
        
        # B. Extract Abundance & Map
        abundance = adata.obsm["means_cell_abundance_w_sf"].copy()
        
        # Create mapping dictionary for columns
        lineage_map = {col: map_to_h6_like(col) for col in abundance.columns}
        
        # C. Aggregate (Sum columns by group)
        h6_abundance = abundance.groupby(lineage_map, axis=1).sum()
        
        # D. Normalize (Proportion per spot)
        # Each row (spot) sums to 100%
        h6_props_spot = h6_abundance.div(h6_abundance.sum(axis=1), axis=0) * 100
        
        # E. Average across the whole sample
        # This creates one row of data for this patient
        avg_props = h6_props_spot.mean(axis=0).to_dict()
        avg_props["SampleID"] = sid
        
        results_list.append(avg_props)
        print(f"[{i+1}/{len(valid_files)}] ✔ Processed {sid}")
        
    except Exception as e:
        print(f"[{i+1}/{len(valid_files)}] ❌ Error processing {sid}: {e}")

# ==============================================================================
# 4. EXPORT FINAL CSV
# ==============================================================================

if results_list:
    final_df = pd.DataFrame(results_list)
    
    # Reorder columns: SampleID first, then alphabetical cell types
    cols = ["SampleID"] + sorted([c for c in final_df.columns if c != "SampleID"])
    final_df = final_df[cols]
    
    # Fill missing values with 0 (e.g., if a sample had NO T-cells)
    final_df = final_df.fillna(0)
    
    out_name = "Spatial_h6_Proportions_Summary.csv"
    final_df.to_csv(out_name, index=False)
    
    print("\n" + "="*40)
    print(f"Done! Saved summary table to: {out_name}")
    print("="*40)
    print(final_df.head())
else:
    print("\nNo results generated. Check file paths or metadata.")

Loading metadata from: /Users/adiallo/Desktop/Thesis/Data_Documents/data_all.csv

Scanning for valid files...
Found 0 valid files to process.

Starting batch processing...

No results generated. Check file paths or metadata.
